In [ ]:
# 1) Dependances
!pip install -q opencv-python yt_dlp kaggle

In [ ]:
# 2) PULL : on remet la VERSION KAGGLE ACTUELLE dans le dossier Drive.
# Chemins ECRITS EN ENTIER (pas de {variable} : evite le bug du dossier "{dataset_path}").
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, shutil
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

# nettoie un eventuel download rate au nom litteral
shutil.rmtree('{dataset_path}', ignore_errors=True)

# vide les classes perimees puis telecharge la version a jour DANS le Drive
!rm -rf "/content/drive/MyDrive/Dataset_Kaggle/Fighter" "/content/drive/MyDrive/Dataset_Kaggle/Platformer" "/content/drive/MyDrive/Dataset_Kaggle/Racing"
!kaggle datasets download maximeclment/datasetpaia3 --unzip -p "/content/drive/MyDrive/Dataset_Kaggle"

for c in ['Fighter', 'Platformer', 'Racing']:
    d = f"/content/drive/MyDrive/Dataset_Kaggle/{c}"
    print(c, ":", len(os.listdir(d)) if os.path.isdir(d) else "ABSENT")
print("PULL termine : dossier Drive = version Kaggle actuelle.")

In [ ]:
# 3) EXTRACTION : on AJOUTE des frames de jeux 2D absents du dataset.
import cv2, os, yt_dlp, time

DOSSIER_BASE_DRIVE = '/content/drive/MyDrive/Dataset_Kaggle'

def extraire_dataset(categorie, nom_du_jeu, url_youtube, intervalle_secondes=15):
    dossier_sortie = os.path.join(DOSSIER_BASE_DRIVE, categorie)
    os.makedirs(dossier_sortie, exist_ok=True)
    fichier_video_temp = f"temp_{nom_du_jeu}.mp4"
    ydl_opts = {
        'format': 'bestvideo[ext=mp4][vcodec^=avc1][height<=720]/best[ext=mp4][height<=720]',
        'outtmpl': fichier_video_temp, 'quiet': False, 'noplaylist': True,
    }
    print(f"\n--- Telechargement : {nom_du_jeu} ---")
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url_youtube])
    except Exception as e:
        print(f"Erreur telechargement : {e}")
        return
    print("--- Extraction des images ---")
    cap = cv2.VideoCapture(fichier_video_temp)
    if not cap.isOpened():
        print("Impossible d'ouvrir la video.")
        return
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    frame_interval = int(fps * intervalle_secondes)
    count = saved_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame = frame[10:-10, 10:-10]  # anti-bandes noires
        if count % frame_interval == 0:
            cv2.imwrite(os.path.join(dossier_sortie, f"{nom_du_jeu}_{saved_count:04d}.jpg"), frame)
            saved_count += 1
            if saved_count % 50 == 0:
                print(f"  {saved_count} images...")
        count += 1
    cap.release()
    if os.path.exists(fichier_video_temp):
        os.remove(fichier_video_temp)
    print(f"OK {nom_du_jeu} : {saved_count} images ajoutees.")

# 7 jeux 2D ABSENTS du dataset (Antoine a retire les platformers 3D)
liste_videos = [
    {"categorie": "Platformer", "jeu": "Rayman_Legends_PC",          "url": "https://www.youtube.com/watch?v=WC8PR2F-xX0"},
    {"categorie": "Platformer", "jeu": "Rayman_Origins_PC",          "url": "https://www.youtube.com/watch?v=lVzzJXcn5OY"},
    {"categorie": "Platformer", "jeu": "Donkey_Kong_Country_2_SNES", "url": "https://www.youtube.com/watch?v=VgTt5fceVTE"},
    {"categorie": "Platformer", "jeu": "Shovel_Knight_PC",           "url": "https://www.youtube.com/watch?v=qeFIyI5Q7Q0"},
    {"categorie": "Platformer", "jeu": "Celeste_PC",                 "url": "https://www.youtube.com/watch?v=gGT02hAWBHE"},
    {"categorie": "Platformer", "jeu": "Sonic_Mania_PC",             "url": "https://www.youtube.com/watch?v=6lhm7ZvAw68"},
    {"categorie": "Platformer", "jeu": "Ori_Blind_Forest_PC",        "url": "https://www.youtube.com/watch?v=BT9NSFWqz1g"},
]

for item in liste_videos:
    extraire_dataset(item["categorie"], item["jeu"], item["url"], intervalle_secondes=15)
    time.sleep(2)

p = os.path.join(DOSSIER_BASE_DRIVE, "Platformer")
print(f"\nPlatformer total apres ajout : {len(os.listdir(p))} images")

In [ ]:
# 4) PUSH : le dossier = version complete + nouvelles frames -> aucune perte.
import os, json
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

meta = {"title": "PA IA3ABD2 Videogame Screenshots",
        "id": "maximeclment/datasetpaia3",
        "licenses": [{"name": "CC0-1.0"}]}
with open("/content/drive/MyDrive/Dataset_Kaggle/dataset-metadata.json", 'w') as f:
    json.dump(meta, f)

!kaggle datasets version -p "/content/drive/MyDrive/Dataset_Kaggle" -m "v12 : +Platformer 2D (Rayman Legends/Origins, DKC2, Shovel Knight, Celeste)" --dir-mode zip